In [0]:
class Silver_races():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('formula1_race.bronze.races')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when,to_timestamp,lit,concat,try_to_timestamp
        apply_tran_df=colrename_df
        for c,t in apply_tran_df.dtypes:
            if t=='string':
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isin('\\N'),None).otherwise(col(c)))
        
        apply_tran_df = apply_tran_df.withColumn("timestamp",
                                            to_timestamp(
                                                concat(col("date") ,lit(" ") , col("time")),
                                                "yyyy-MM-dd HH:mm:ss"
                                            ))
        apply_tran_df= (apply_tran_df.selectExpr('race_id','year as race_year','round','circuit_id','name as race_name',
                                                 'date as race_date','time as race_time','timestamp','races_ingestion_date','source'
                        ))
        
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("formula1_race.silver.races")
        display(spark.sql(f"select count(*) from formula1_race.silver.races"))
        print("Data write into sliver races table is Done")
        
    
        
     
    def process(self):
        print("Started silver-ingestion-races  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
        


In [0]:
Silver_races_instance = Silver_races("races")
Silver_races_instance.process()